In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Load the cleaned data
df = pd.read_csv('../data/processed/flights_clean.csv')
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
df.head()

Total rows: 2,913,802
Total columns: 34


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,DELAYED,DEP_HOUR
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN,0,11
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN,0,21
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN,0,9
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0,1,16
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN,0,18


In [5]:
# Null counts per column
print("── Null counts ──")
print(df.isnull().sum().sort_values(ascending=False).head(15))

── Null counts ──
CANCELLATION_CODE          2913802
DELAY_DUE_LATE_AIRCRAFT    2379939
DELAY_DUE_NAS              2379939
DELAY_DUE_SECURITY         2379939
DELAY_DUE_CARRIER          2379939
DELAY_DUE_WEATHER          2379939
AIRLINE                          0
FL_DATE                          0
AIRLINE_DOT                      0
AIRLINE_CODE                     0
CRS_DEP_TIME                     0
DEP_TIME                         0
DOT_CODE                         0
FL_NUMBER                        0
ORIGIN                           0
dtype: int64


In [6]:
# Delay rate by airline
delay_by_airline = df.groupby('AIRLINE_CODE')['DELAYED'].mean().sort_values(ascending=False).reset_index()
delay_by_airline.columns = ['Airline', 'Delay Rate']
delay_by_airline['Delay Rate'] = (delay_by_airline['Delay Rate'] * 100).round(2)

print("── Delay Rate by Airline (%) ──")
print(delay_by_airline.to_string(index=False))


── Delay Rate by Airline (%) ──
Airline  Delay Rate
     B6       27.00
     F9       26.65
     G4       26.09
     NK       22.12
     EV       21.48
     AA       19.90
     WN       19.30
     UA       19.16
     YV       18.63
     AS       18.18
     MQ       17.39
     OH       17.35
     HA       16.38
     OO       15.89
     YX       15.80
     QX       14.51
     DL       14.41
     9E       13.04


In [7]:
# Average arrival delay by hour of day
delay_by_hour = df.groupby('DEP_HOUR')['ARR_DELAY'].mean().reset_index()
delay_by_hour.columns = ['Hour', 'Avg Arrival Delay (min)']

plt.figure(figsize=(12, 5))
plt.bar(delay_by_hour['Hour'], delay_by_hour['Avg Arrival Delay (min)'], color='steelblue')
plt.xlabel('Departure Hour')
plt.ylabel('Avg Arrival Delay (min)')
plt.title('Average Arrival Delay by Departure Hour')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig('../docs/delay_by_hour.png')
plt.show()
print("✓ Saved chart")

✓ Saved chart


C:\Users\Ameer\AppData\Local\Temp\ipykernel_22732\1496125002.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Average arrival delay by day of week
df['DAY_OF_WEEK'] = pd.to_datetime(df['FL_DATE']).dt.dayofweek
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

delay_by_day = df.groupby('DAY_OF_WEEK')['ARR_DELAY'].mean().reset_index()
delay_by_day['DAY_NAME'] = delay_by_day['DAY_OF_WEEK'].map(lambda x: day_names[x])

plt.figure(figsize=(8, 5))
plt.bar(delay_by_day['DAY_NAME'], delay_by_day['ARR_DELAY'], color='coral')
plt.xlabel('Day of Week')
plt.ylabel('Avg Arrival Delay (min)')
plt.title('Average Arrival Delay by Day of Week')
plt.tight_layout()
plt.savefig('../docs/delay_by_day.png')
plt.show()
print("✓ Saved chart")


✓ Saved chart


C:\Users\Ameer\AppData\Local\Temp\ipykernel_22732\2201696229.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Top 10 airports by average arrival delay
delay_by_airport = df.groupby('ORIGIN')['ARR_DELAY'].mean().sort_values(ascending=False).head(10).reset_index()
delay_by_airport.columns = ['Airport', 'Avg Arrival Delay (min)']

plt.figure(figsize=(10, 5))
plt.barh(delay_by_airport['Airport'][::-1], delay_by_airport['Avg Arrival Delay (min)'][::-1], color='tomato')
plt.xlabel('Avg Arrival Delay (min)')
plt.title('Top 10 Airports by Average Arrival Delay')
plt.tight_layout()
plt.savefig('../docs/delay_by_airport.png')
plt.show()
print("✓ Saved chart")
print(delay_by_airport.to_string(index=False))


✓ Saved chart
Airport  Avg Arrival Delay (min)
    PPG                56.451613
    SMX                37.895349
    PSM                31.115854
    PGV                24.785714
    HGR                24.752809
    HTS                22.650000
    ASE                22.213898
    HOB                21.630682
    SCK                20.818182
    BLV                20.629259


C:\Users\Ameer\AppData\Local\Temp\ipykernel_22732\514148940.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Delayed vs not delayed distribution
counts = df['DELAYED'].value_counts()
labels = ['On Time', 'Delayed (≥15 min)']

plt.figure(figsize=(6, 5))
plt.bar(labels, counts.values, color=['steelblue', 'tomato'])
plt.ylabel('Number of Flights')
plt.title('Flight Delay Distribution')
plt.tight_layout()
plt.savefig('../docs/delay_distribution.png')
plt.show()

print(f"On time: {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)")
print(f"Delayed: {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)")
print("✓ Saved chart")


On time: 2,379,939 (81.7%)
Delayed: 533,863 (18.3%)
✓ Saved chart


C:\Users\Ameer\AppData\Local\Temp\ipykernel_22732\3344304740.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
